# Batch 3 — Training (Regularization + Threshold Tuning)

**Part A — Regularization refinement.** A narrow re-search around Batch 1's
winning hyperparameters for XGBoost, LightGBM, and Logistic Regression,
explicitly adding L1/L2 regularization strength (`reg_alpha`/`reg_lambda`
for the boosters, a finer `C` grid for Logistic Regression) — the one
lever Batch 1's exploratory search didn't target directly. Whichever
configuration wins on CV PR-AUC (Batch 1's original or this refinement)
is what continues to Part B.

**Part B — Threshold tuning (the main focus of this batch).** A brief note
on what "the sweet spot" means here: precision and recall trade off
against each other by construction (once you fix a model, raising one
lowers the other), so no single threshold maximizes both *simultaneously*
— the practical version of "find the sweet spot" is **maximizing F1**, the
harmonic mean of precision and recall, which is the standard way to
express "the best balance of both." We compute the full precision/recall
curve so the trade-off is visible, mark the F1-optimal point, and also
report an F2-optimal point (weights recall higher — relevant if missing
fraud is costlier than a false accusation) and an F0.5-optimal point
(weights precision higher) as alternative operating points, since which
balance is "right" is a business decision, not a purely statistical one.

**Threshold selection methodology:** the threshold is chosen using
**out-of-fold cross-validated probabilities on the training set**
(`cross_val_predict`), never the validation set — exactly the same
train/validate discipline used throughout this project. The chosen
threshold is then applied to the validation set once, to report an honest,
unbiased estimate of what it achieves.


In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths & Load Artifacts

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_3' else Path.cwd()
BATCH1_DIR = PROJECT_ROOT / 'Models_Batch_1'
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_3'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'
MODELS_DIR = ARTIFACTS_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

setup = joblib.load(ARTIFACTS_DIR / 'batch3_setup.joblib')
FINALIST_MODELS = setup['FINALIST_MODELS']
REGULARIZATION_REFINE_MODELS = setup['REGULARIZATION_REFINE_MODELS']
FEATURE_SET_MAP = setup['FEATURE_SET_MAP']
y_train, y_val = setup['y_train'], setup['y_val']

batch1_results = pd.read_csv(BATCH1_DIR / 'artifacts' / 'batch1_results.csv')
batch2_results = pd.read_csv(PROJECT_ROOT / 'Models_Batch_2' / 'artifacts' / 'batch2_results.csv')
batch2_results['Sampling_Technique'] = batch2_results['Sampling_Technique'].fillna('None')

BATCH1_MODEL_FILES = {
    'XGBoost': 'XGBoost.joblib', 'LightGBM': 'LightGBM.joblib',
    'LogisticRegression': 'LogisticRegression_rob.joblib',
    'MLP': 'MLP.joblib', 'RandomForest': 'RandomForest.joblib',
}
batch1_models = {name: joblib.load(BATCH1_DIR / 'artifacts' / 'models' / f)
                  for name, f in BATCH1_MODEL_FILES.items()}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {'pr_auc': 'average_precision', 'recall': 'recall', 'precision': 'precision',
           'f1': 'f1', 'roc_auc': 'roc_auc'}
print("Loaded setup, Batch 1 results/models, Batch 2 results.")


Loaded setup, Batch 1 results/models, Batch 2 results.


## 2. Part A — Regularization-Focused Hyperparameter Refinement

For each model, a narrow `RandomizedSearchCV`/`GridSearchCV` centered on
Batch 1's winning configuration, checked with `try/except` + a results
checkpoint (the same resilience pattern Batch 2 needed after its own
crashes) so a failure or a future rerun doesn't lose completed work.


In [3]:
refine_results = []
CHECKPOINT_PATH = ARTIFACTS_DIR / 'batch3_refine_checkpoint.csv'
if CHECKPOINT_PATH.exists():
    refine_results = pd.read_csv(CHECKPOINT_PATH).to_dict('records')
    print(f"Resumed refinement checkpoint: {len(refine_results)} model(s) already done.")
done_refine = {r['Model'] for r in refine_results}

def run_refinement(name, estimator, param_space, X_train, search_type='random', n_iter=25):
    if name in done_refine:
        print(f"[skip] {name} refinement — already in checkpoint")
        return
    print(f"\n{'='*70}\nRefining: {name}\n{'='*70}")
    t0 = time.time()
    try:
        if search_type == 'random':
            search = RandomizedSearchCV(estimator, param_space, n_iter=n_iter, cv=cv5,
                                         scoring=SCORING, refit='pr_auc', random_state=RANDOM_STATE, n_jobs=1)
        else:
            search = GridSearchCV(estimator, param_space, cv=cv5, scoring=SCORING, refit='pr_auc', n_jobs=1)
        search.fit(X_train, y_train)
        elapsed = time.time() - t0
        idx = search.best_index_
        cv_pr_auc = search.cv_results_['mean_test_pr_auc'][idx]

        b1_cv_pr_auc = batch1_results.loc[batch1_results['Model'] == (
            'LogisticRegression_rob' if name == 'LogisticRegression' else name), 'CV_PR_AUC'].values[0]
        improved = cv_pr_auc > b1_cv_pr_auc

        print(f"Batch 1 CV PR-AUC: {b1_cv_pr_auc:.4f}  |  Batch 3 refined CV PR-AUC: {cv_pr_auc:.4f}  |  "
              f"{'IMPROVED' if improved else 'no improvement'}")
        print(f"Best params: {search.best_params_}")
        print(f"Elapsed: {elapsed:.1f}s")

        joblib.dump(search.best_estimator_, MODELS_DIR / f'{name}_refined.joblib')
        refine_results.append({
            'Model': name, 'Batch1_CV_PR_AUC': round(b1_cv_pr_auc, 4),
            'Batch3_CV_PR_AUC': round(cv_pr_auc, 4), 'Improved': improved,
            'Best_Params': str(search.best_params_), 'Train_Time_s': round(elapsed, 1),
        })
    except Exception as e:
        elapsed = time.time() - t0
        print(f"FAILED after {elapsed:.1f}s: {type(e).__name__}: {e}")
        refine_results.append({
            'Model': name, 'Batch1_CV_PR_AUC': np.nan, 'Batch3_CV_PR_AUC': np.nan,
            'Improved': False, 'Best_Params': f'FAILED: {e}', 'Train_Time_s': round(elapsed, 1),
        })
    finally:
        pd.DataFrame(refine_results).to_csv(CHECKPOINT_PATH, index=False)


Resumed refinement checkpoint: 3 model(s) already done.


In [4]:
# XGBoost: narrow search around Batch 1's winner (subsample=0.7, n_estimators=200, max_depth=3,
# learning_rate=0.1, colsample_bytree=0.7), adding reg_alpha/reg_lambda
xgb_params = {
    'learning_rate': [0.03, 0.05, 0.07, 0.1, 0.15],
    'n_estimators': [150, 200, 250, 300],
    'max_depth': [3, 4],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.01, 0.1, 1, 5],
    'reg_lambda': [0.1, 1, 5, 10, 20],
}
run_refinement('XGBoost',
                XGBClassifier(eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=2),
                xgb_params, setup['X_train_tree'], search_type='random', n_iter=25)


[skip] XGBoost refinement — already in checkpoint


In [5]:
# LightGBM: narrow search around Batch 1's winner (subsample=0.7, num_leaves=63, n_estimators=100,
# max_depth=5, learning_rate=0.1, colsample_bytree=0.7), adding reg_alpha/reg_lambda + min_child_samples
# (min_child_samples is a complexity-reduction lever per the project plan)
lgbm_params = {
    'learning_rate': [0.03, 0.05, 0.07, 0.1, 0.15],
    'n_estimators': [80, 100, 150, 200],
    'num_leaves': [31, 63, 90],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.01, 0.1, 1, 5],
    'reg_lambda': [0, 0.1, 1, 5, 10],
}
run_refinement('LightGBM',
                LGBMClassifier(random_state=RANDOM_STATE, n_jobs=2, verbosity=-1),
                lgbm_params, setup['X_train_tree'], search_type='random', n_iter=25)


[skip] LightGBM refinement — already in checkpoint


In [6]:
# Logistic Regression: finer C grid around Batch 1's winner (C=0.1, penalty='l1'), checking L2 too
logreg_params = {
    'C': np.logspace(-3, 1, 25),
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
}
run_refinement('LogisticRegression',
                LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
                logreg_params, setup['X_train_rob'], search_type='grid')


[skip] LogisticRegression refinement — already in checkpoint


## 3. Part A Results — Did Regularization Refinement Beat Batch 1?

In [7]:
refine_df = pd.DataFrame(refine_results)
refine_df.to_csv(ARTIFACTS_DIR / 'batch3_refinement_results.csv', index=False)
refine_df


,Model,Batch1_CV_PR_AUC,Batch3_CV_PR_AUC,Improved,Best_Params,Train_Time_s
0,XGBoost,0.8037,0.8040,True,"{'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha...",82.7
1,LightGBM,0.8006,0.8008,True,"{'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha...",72.9
2,LogisticRegression,0.7255,0.7262,True,"{'C': 0.03162277660168379, 'penalty': 'l1', 's...",138.9


## 4. Assemble the Final Model Per Finalist

For XGBoost/LightGBM/Logistic Regression: use the Batch 3 refined model if
(and only if) it beat Batch 1's CV PR-AUC, otherwise keep Batch 1's
original. For MLP/Random Forest (not re-tuned): keep Batch 1's original.


In [8]:
final_models = {}
for name in FINALIST_MODELS:
    if name in REGULARIZATION_REFINE_MODELS:
        row = refine_df[refine_df['Model'] == name].iloc[0]
        if row['Improved']:
            final_models[name] = joblib.load(MODELS_DIR / f'{name}_refined.joblib')
            print(f"{name}: using Batch 3 REFINED model (CV PR-AUC {row['Batch3_CV_PR_AUC']:.4f} > "
                  f"Batch 1's {row['Batch1_CV_PR_AUC']:.4f})")
        else:
            final_models[name] = batch1_models[name]
            print(f"{name}: keeping Batch 1 ORIGINAL model (refinement did not improve CV PR-AUC)")
    else:
        final_models[name] = batch1_models[name]
        print(f"{name}: keeping Batch 1 ORIGINAL model (not re-tuned this batch)")


XGBoost: using Batch 3 REFINED model (CV PR-AUC 0.8040 > Batch 1's 0.8037)


LightGBM: using Batch 3 REFINED model (CV PR-AUC 0.8008 > Batch 1's 0.8006)
LogisticRegression: using Batch 3 REFINED model (CV PR-AUC 0.7262 > Batch 1's 0.7255)
MLP: keeping Batch 1 ORIGINAL model (not re-tuned this batch)
RandomForest: keeping Batch 1 ORIGINAL model (not re-tuned this batch)


## 5. Part B — Threshold Tuning

For each finalist, get honest out-of-fold probabilities on the training
set via `cross_val_predict` (a fresh clone of the model is fit per fold
internally — the already-fit `final_models[name]` is only used as a
template for its hyperparameters), sweep every possible threshold from the
precision-recall curve, and locate the F1-optimal (balanced), F2-optimal
(recall-leaning), and F0.5-optimal (precision-leaning) thresholds.


In [9]:
def best_threshold_by_fbeta(y_true, proba, beta):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # precision_recall_curve returns one more point than thresholds (the (1, 0) endpoint
    # with no corresponding threshold) — drop it so arrays align
    precision, recall = precision[:-1], recall[:-1]
    with np.errstate(divide='ignore', invalid='ignore'):
        fbeta = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)
    fbeta = np.nan_to_num(fbeta, nan=0.0)
    best_idx = np.argmax(fbeta)
    return thresholds[best_idx], precision[best_idx], recall[best_idx], fbeta[best_idx]


In [10]:
threshold_results = []
THRESH_CHECKPOINT = ARTIFACTS_DIR / 'batch3_threshold_checkpoint.csv'
if THRESH_CHECKPOINT.exists():
    threshold_results = pd.read_csv(THRESH_CHECKPOINT).to_dict('records')
    print(f"Resumed threshold checkpoint: {len(threshold_results)} row(s) already done.")
done_threshold = {(r['Model'], r['Metric']) for r in threshold_results}

for name in FINALIST_MODELS:
    key_check = (name, 'F1')
    if key_check in done_threshold:
        print(f"[skip] {name} — already thresholded")
        continue

    print(f"\n{'='*70}\n{name}\n{'='*70}")
    t0 = time.time()
    try:
        fset = FEATURE_SET_MAP[name]
        X_train_m = setup[f'X_train_{fset}']
        X_val_m = setup[f'X_val_{fset}']

        model_template = clone(final_models[name])
        oof_proba = cross_val_predict(model_template, X_train_m, y_train, cv=cv5,
                                       method='predict_proba', n_jobs=1)[:, 1]
        print(f"Out-of-fold CV probabilities collected ({time.time()-t0:.1f}s)")

        val_proba = final_models[name].predict_proba(X_val_m)[:, 1]

        for metric_name, beta in [('F0.5', 0.5), ('F1', 1.0), ('F2', 2.0)]:
            thresh, oof_prec, oof_rec, oof_fbeta = best_threshold_by_fbeta(y_train, oof_proba, beta)

            val_pred = (val_proba >= thresh).astype(int)
            val_prec = precision_score(y_val, val_pred, zero_division=0)
            val_rec = recall_score(y_val, val_pred)
            val_f1 = f1_score(y_val, val_pred, zero_division=0)
            val_fbeta = fbeta_score(y_val, val_pred, beta=beta, zero_division=0)

            print(f"  [{metric_name}-optimal] threshold={thresh:.3f} | "
                  f"OOF(train): P={oof_prec:.3f} R={oof_rec:.3f} | "
                  f"Val: P={val_prec:.3f} R={val_rec:.3f} F1={val_f1:.3f} F{beta}={val_fbeta:.3f}")

            threshold_results.append({
                'Model': name, 'Metric': metric_name, 'Threshold': round(thresh, 4),
                'OOF_Train_Precision': round(oof_prec, 4), 'OOF_Train_Recall': round(oof_rec, 4),
                'Val_Precision': round(val_prec, 4), 'Val_Recall': round(val_rec, 4),
                'Val_F1': round(val_f1, 4), 'Val_Fbeta': round(val_fbeta, 4),
            })

        # also record the default 0.5 threshold for direct comparison
        val_pred_default = (val_proba >= 0.5).astype(int)
        threshold_results.append({
            'Model': name, 'Metric': 'Default_0.5', 'Threshold': 0.5,
            'OOF_Train_Precision': np.nan, 'OOF_Train_Recall': np.nan,
            'Val_Precision': round(precision_score(y_val, val_pred_default, zero_division=0), 4),
            'Val_Recall': round(recall_score(y_val, val_pred_default), 4),
            'Val_F1': round(f1_score(y_val, val_pred_default, zero_division=0), 4),
            'Val_Fbeta': np.nan,
        })
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        threshold_results.append({
            'Model': name, 'Metric': 'F1', 'Threshold': np.nan,
            'OOF_Train_Precision': np.nan, 'OOF_Train_Recall': np.nan,
            'Val_Precision': np.nan, 'Val_Recall': np.nan, 'Val_F1': np.nan, 'Val_Fbeta': np.nan,
        })
    finally:
        pd.DataFrame(threshold_results).to_csv(THRESH_CHECKPOINT, index=False)


Resumed threshold checkpoint: 20 row(s) already done.
[skip] XGBoost — already thresholded
[skip] LightGBM — already thresholded
[skip] LogisticRegression — already thresholded
[skip] MLP — already thresholded
[skip] RandomForest — already thresholded


## 6. Save Threshold Results & Final Models

In [11]:
threshold_df = pd.DataFrame(threshold_results)
threshold_df.to_csv(ARTIFACTS_DIR / 'batch3_threshold_results.csv', index=False)

for name, model in final_models.items():
    joblib.dump(model, MODELS_DIR / f'{name}_final.joblib')

print(f"Saved {len(threshold_df)} threshold rows and {len(final_models)} final models.")
threshold_df.sort_values(['Model', 'Metric'])


Saved 20 threshold rows and 5 final models.


,Model,Metric,Threshold,OOF_Train_Precision,OOF_Train_Recall,Val_Precision,Val_Recall,Val_F1,Val_Fbeta
7,LightGBM,Default_0.5,0.5000,NaN,NaN,0.9040,0.5970,0.7191,NaN
4,LightGBM,F0.5,0.6608,0.9424,0.5750,0.9568,0.5510,0.6993,0.8340
5,LightGBM,F1,0.3775,0.8407,0.6597,0.8602,0.6430,0.7359,0.7359
6,LightGBM,F2,0.1624,0.6699,0.7445,0.6693,0.7351,0.7007,0.7209
11,LogisticRegression,Default_0.5,0.5000,NaN,NaN,0.8840,0.4266,0.5755,NaN
8,LogisticRegression,F0.5,0.3391,0.8376,0.5414,0.8410,0.5261,0.6473,0.7511
9,LogisticRegression,F1,0.2347,0.7444,0.6427,0.7462,0.6070,0.6694,0.6694
10,LogisticRegression,F2,0.1587,0.5770,0.7426,0.5626,0.7152,0.6298,0.6784
15,MLP,Default_0.5,0.5000,NaN,NaN,0.8932,0.6032,0.7201,NaN
12,MLP,F0.5,0.6634,0.9493,0.5590,0.9603,0.5410,0.6921,0.8314


## 5b. Extra Operating Point — F1.5 (recall weighted moderately above precision)

The F1-optimal point above still gives precision noticeably higher than recall,
and F2-optimal swings hard the other way (precision drops to ~0.65). Per a
business call to prioritize catching fraud while keeping precision at a level
still considered acceptable (not "worst-case"), we add one more point on the
same curve: **beta=1.5**, sitting between F1 and F2. Same discipline as above —
threshold chosen from out-of-fold probabilities on the training set only, then
applied once to validation. Test set is intentionally not touched here.


In [12]:
extra_threshold_results = []
EXTRA_MODELS = ['XGBoost', 'LightGBM']  # the two finalists carried into Final_Report
EXTRA_BETAS = [('F1.3', 1.3), ('F1.5', 1.5), ('F1.75', 1.75)]  # scan the zone between F1 and F2

for name in EXTRA_MODELS:
    fset = FEATURE_SET_MAP[name]
    X_train_m = setup[f'X_train_{fset}']
    X_val_m = setup[f'X_val_{fset}']

    model_template = clone(final_models[name])
    oof_proba = cross_val_predict(model_template, X_train_m, y_train, cv=cv5,
                                   method='predict_proba', n_jobs=1)[:, 1]
    val_proba = final_models[name].predict_proba(X_val_m)[:, 1]

    for metric_name, beta in EXTRA_BETAS:
        thresh, oof_prec, oof_rec, oof_fbeta = best_threshold_by_fbeta(y_train, oof_proba, beta)

        val_pred = (val_proba >= thresh).astype(int)
        val_prec = precision_score(y_val, val_pred, zero_division=0)
        val_rec = recall_score(y_val, val_pred)
        val_f1 = f1_score(y_val, val_pred, zero_division=0)
        val_fbeta = fbeta_score(y_val, val_pred, beta=beta, zero_division=0)

        print(f"{name} [{metric_name}-optimal] threshold={thresh:.3f} | "
              f"OOF(train): P={oof_prec:.3f} R={oof_rec:.3f} | "
              f"Val: P={val_prec:.3f} R={val_rec:.3f} F1={val_f1:.3f} F{beta}={val_fbeta:.3f}")

        extra_threshold_results.append({
            'Model': name, 'Metric': metric_name, 'Threshold': round(thresh, 4),
            'OOF_Train_Precision': round(oof_prec, 4), 'OOF_Train_Recall': round(oof_rec, 4),
            'Val_Precision': round(val_prec, 4), 'Val_Recall': round(val_rec, 4),
            'Val_F1': round(val_f1, 4), 'Val_Fbeta': round(val_fbeta, 4),
        })

extra_df = pd.DataFrame(extra_threshold_results)

# de-dupe in case this cell is re-run, then persist alongside the F0.5/F1/F2/Default rows
extra_metric_names = [m for m, _ in EXTRA_BETAS]
threshold_df = threshold_df[~(threshold_df['Model'].isin(EXTRA_MODELS) & threshold_df['Metric'].isin(extra_metric_names))]
threshold_df = pd.concat([threshold_df, extra_df], ignore_index=True)
threshold_df.to_csv(ARTIFACTS_DIR / 'batch3_threshold_results.csv', index=False)
extra_df


XGBoost [F1.3-optimal] threshold=0.276 | OOF(train): P=0.773 R=0.707 | Val: P=0.783 R=0.682 F1=0.729 F1.3=0.716
XGBoost [F1.5-optimal] threshold=0.231 | OOF(train): P=0.734 R=0.725 | Val: P=0.736 R=0.700 F1=0.718 F1.5=0.711
XGBoost [F1.75-optimal] threshold=0.195 | OOF(train): P=0.689 R=0.743 | Val: P=0.682 R=0.724 F1=0.702 F1.75=0.713


LightGBM [F1.3-optimal] threshold=0.253 | OOF(train): P=0.769 R=0.703 | Val: P=0.772 R=0.690 F1=0.729 F1.3=0.719
LightGBM [F1.5-optimal] threshold=0.238 | OOF(train): P=0.757 R=0.709 | Val: P=0.755 R=0.694 F1=0.723 F1.5=0.712
LightGBM [F1.75-optimal] threshold=0.162 | OOF(train): P=0.670 R=0.744 | Val: P=0.669 R=0.735 F1=0.701 F1.75=0.718


,Model,Metric,Threshold,OOF_Train_Precision,OOF_Train_Recall,Val_Precision,Val_Recall,Val_F1,Val_Fbeta
0,XGBoost,F1.3,0.2756,0.7727,0.7066,0.7829,0.6816,0.7287,0.7160
1,XGBoost,F1.5,0.2315,0.7343,0.7253,0.7359,0.7002,0.7177,0.7109
2,XGBoost,F1.75,0.1953,0.6889,0.7434,0.6823,0.7239,0.7025,0.7132
3,LightGBM,F1.3,0.2535,0.7692,0.7032,0.7719,0.6903,0.7288,0.7185
4,LightGBM,F1.5,0.2384,0.7570,0.7088,0.7551,0.6940,0.7233,0.7117
5,LightGBM,F1.75,0.1624,0.6699,0.7445,0.6693,0.7351,0.7007,0.7177


---
**Next:** `evaluation.ipynb` — plots the precision-recall-vs-threshold
curves with the F1/F2/F0.5-optimal points marked, directly compares the
tuned-threshold recall/precision against Batch 2's best sampling-technique
recall/precision (testing the "threshold tuning is a cleaner lever than
resampling" hypothesis), updates the master comparison table, and
recommends a final model + threshold.
